In [ ]:
# ============================================================
# STEP 4: VaR FORECASTING AND BACKTESTING
# Step 4A: In-sample VaR backtesting
# Step 4B: Out-of-sample rolling VaR backtesting
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import chi2
from arch import arch_model
from arch.univariate import Normal, StudentsT, SkewStudent

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROCESSED_DIR = Path("data/processed")
OUTPUT_DIR = Path("outputs")
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
VAR_DIR = OUTPUT_DIR / "var_backtesting"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
VAR_DIR.mkdir(parents=True, exist_ok=True)

RETURNS_PATH = PROCESSED_DIR / "vietnam_size_indices_log_returns_common_20141121_20251231.csv"

if not RETURNS_PATH.exists():
    RETURNS_PATH = Path("/mnt/data/vietnam_size_indices_log_returns_common_20141121_20251231.csv")

# ------------------------------------------------------------
# Load returns
# ------------------------------------------------------------

returns = pd.read_csv(RETURNS_PATH, parse_dates=["date"])
returns = returns.set_index("date").sort_index()

INDEX_COLUMNS = ["VN30", "VNAllshare", "VNMidcap", "VNSmallcap"]
returns = returns[INDEX_COLUMNS]

assert returns.isna().sum().sum() == 0
assert returns.index.is_monotonic_increasing
assert returns.index.duplicated().sum() == 0
assert np.isfinite(returns.to_numpy()).all()

print("Return data loaded.")
print("Sample:", returns.index.min().date(), "to", returns.index.max().date())
print("Shape:", returns.shape)
display(returns.head())

In [ ]:
# ============================================================
# 1. BALANCED MODEL SET
# ============================================================

# ============================================================
# FULL MODEL SPECS FOR STEP 4
# Includes Student-t and Skewed Student-t variants
# ============================================================

MODEL_SPECS = {
    "AR1_GARCH11_Normal": {
        "mean": "AR", "lags": 1,
        "vol": "GARCH", "p": 1, "o": 0, "q": 1,
        "dist": "normal"
    },

    "AR1_GARCH11_StudentT": {
        "mean": "AR", "lags": 1,
        "vol": "GARCH", "p": 1, "o": 0, "q": 1,
        "dist": "t"
    },

    "AR1_GARCH11_SkewT": {
        "mean": "AR", "lags": 1,
        "vol": "GARCH", "p": 1, "o": 0, "q": 1,
        "dist": "skewt"
    },

    "AR1_GJR_GARCH11_StudentT": {
        "mean": "AR", "lags": 1,
        "vol": "GARCH", "p": 1, "o": 1, "q": 1,
        "dist": "t"
    },

    "AR1_GJR_GARCH11_SkewT": {
        "mean": "AR", "lags": 1,
        "vol": "GARCH", "p": 1, "o": 1, "q": 1,
        "dist": "skewt"
    },

    "AR1_EGARCH11_StudentT": {
        "mean": "AR", "lags": 1,
        "vol": "EGARCH", "p": 1, "o": 1, "q": 1,
        "dist": "t"
    },

    "AR1_EGARCH11_SkewT": {
        "mean": "AR", "lags": 1,
        "vol": "EGARCH", "p": 1, "o": 1, "q": 1,
        "dist": "skewt"
    },

    "AR1_APARCH11_StudentT": {
        "mean": "AR", "lags": 1,
        "vol": "APARCH", "p": 1, "o": 1, "q": 1,
        "dist": "t"
    },

    "AR1_APARCH11_SkewT": {
        "mean": "AR", "lags": 1,
        "vol": "APARCH", "p": 1, "o": 1, "q": 1,
        "dist": "skewt"
    }
}
MODELS_TO_RUN = list(MODEL_SPECS.keys())

print("Models to run:")
for m in MODELS_TO_RUN:
    print("-", m)

missing_specs = [m for m in MODELS_TO_RUN if m not in MODEL_SPECS]
assert len(missing_specs) == 0, f"Missing specs: {missing_specs}"

print("Models used in Step 4:")
for m in MODELS_TO_RUN:
    print("-", m)

In [ ]:
# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

VAR_LEVELS = [0.01, 0.05]
POSITIONS = ["long", "short"]


def make_arch_model(y, spec):
    return arch_model(
        y,
        mean=spec["mean"],
        lags=spec["lags"],
        vol=spec["vol"],
        p=spec["p"],
        o=spec["o"],
        q=spec["q"],
        dist=spec["dist"],
        rescale=False
    )


def fit_one_model(y, spec):
    am = make_arch_model(y, spec)
    res = am.fit(
        update_freq=0,
        disp="off",
        show_warning=False,
        options={"maxiter": 2000}
    )
    return res


def get_distribution_quantile(params, dist_name, prob):
    """
    Return standardized innovation quantile for arch distributions.
    prob: e.g. 0.01, 0.05, 0.95, 0.99
    """
    prob = float(prob)

    if dist_name == "normal":
        return float(Normal().ppf(prob))

    if dist_name == "t":
        nu = params["nu"]
        return float(StudentsT().ppf(prob, parameters=np.array([nu])))

    if dist_name == "skewt":
        eta = params["eta"]
        lam = params["lambda"]
        return float(SkewStudent().ppf(prob, parameters=np.array([eta, lam])))

    raise ValueError(f"Unsupported distribution: {dist_name}")


def safe_log_term(x, p):
    """
    Return x * log(p), safely handling x = 0.
    """
    if x == 0:
        return 0.0
    p = min(max(p, 1e-12), 1 - 1e-12)
    return x * np.log(p)


def kupiec_test(violations, alpha):
    """
    Kupiec unconditional coverage test.
    H0: empirical failure rate = alpha
    """
    v = np.asarray(violations).astype(int)
    T = len(v)
    N = int(v.sum())

    if T == 0:
        return {
            "n_forecasts": 0,
            "n_violations": np.nan,
            "failure_rate": np.nan,
            "expected_violations": np.nan,
            "lr_uc": np.nan,
            "kupiec_pvalue": np.nan,
            "kupiec_pass_5pct": False
        }

    phat = N / T

    ll_null = safe_log_term(T - N, 1 - alpha) + safe_log_term(N, alpha)
    ll_alt = safe_log_term(T - N, 1 - phat) + safe_log_term(N, phat)

    lr_uc = -2 * (ll_null - ll_alt)
    pvalue = 1 - chi2.cdf(lr_uc, df=1)

    return {
        "n_forecasts": T,
        "n_violations": N,
        "failure_rate": phat,
        "expected_violations": alpha * T,
        "lr_uc": lr_uc,
        "kupiec_pvalue": pvalue,
        "kupiec_pass_5pct": pvalue > 0.05
    }


def christoffersen_test(violations, alpha):
    """
    Christoffersen independence and conditional coverage tests.
    H0 independence: violations are independent.
    H0 conditional coverage: correct coverage + independence.
    """
    v = np.asarray(violations).astype(int)

    if len(v) < 2:
        return {
            "n00": np.nan, "n01": np.nan, "n10": np.nan, "n11": np.nan,
            "lr_ind": np.nan, "christoffersen_ind_pvalue": np.nan,
            "lr_cc": np.nan, "christoffersen_cc_pvalue": np.nan,
            "christoffersen_ind_pass_5pct": False,
            "christoffersen_cc_pass_5pct": False
        }

    prev = v[:-1]
    curr = v[1:]

    n00 = int(((prev == 0) & (curr == 0)).sum())
    n01 = int(((prev == 0) & (curr == 1)).sum())
    n10 = int(((prev == 1) & (curr == 0)).sum())
    n11 = int(((prev == 1) & (curr == 1)).sum())

    pi = (n01 + n11) / max(n00 + n01 + n10 + n11, 1)
    pi0 = n01 / max(n00 + n01, 1)
    pi1 = n11 / max(n10 + n11, 1)

    ll_null = (
        safe_log_term(n00 + n10, 1 - pi)
        + safe_log_term(n01 + n11, pi)
    )

    ll_alt = (
        safe_log_term(n00, 1 - pi0)
        + safe_log_term(n01, pi0)
        + safe_log_term(n10, 1 - pi1)
        + safe_log_term(n11, pi1)
    )

    lr_ind = -2 * (ll_null - ll_alt)
    ind_pvalue = 1 - chi2.cdf(lr_ind, df=1)

    kupiec = kupiec_test(v, alpha)
    lr_cc = kupiec["lr_uc"] + lr_ind
    cc_pvalue = 1 - chi2.cdf(lr_cc, df=2)

    return {
        "n00": n00,
        "n01": n01,
        "n10": n10,
        "n11": n11,
        "lr_ind": lr_ind,
        "christoffersen_ind_pvalue": ind_pvalue,
        "lr_cc": lr_cc,
        "christoffersen_cc_pvalue": cc_pvalue,
        "christoffersen_ind_pass_5pct": ind_pvalue > 0.05,
        "christoffersen_cc_pass_5pct": cc_pvalue > 0.05
    }


def dq_test(violations, var_forecast, alpha, lags=5):
    """
    Dynamic Quantile test.

    Regression:
    hit_t = beta0 + beta1 * VaR_t + beta2 * hit_{t-1} + ... + error_t

    hit_t = violation_t - alpha

    H0: all regression coefficients are zero.
    """
    v = pd.Series(np.asarray(violations).astype(int)).reset_index(drop=True)
    var_series = pd.Series(np.asarray(var_forecast, dtype=float)).reset_index(drop=True)

    hit = v.astype(float) - float(alpha)

    df = pd.DataFrame({
        "hit": hit,
        "var": var_series
    })

    for lag in range(1, lags + 1):
        df[f"hit_lag_{lag}"] = df["hit"].shift(lag)

    df = df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

    if len(df) <= lags + 2:
        return {
            "dq_lags": lags,
            "dq_stat": np.nan,
            "dq_df": np.nan,
            "dq_pvalue": np.nan,
            "dq_pass_5pct": False
        }

    y = df["hit"].to_numpy(dtype=float).reshape(-1, 1)

    X_cols = ["var"] + [f"hit_lag_{lag}" for lag in range(1, lags + 1)]
    X = df[X_cols].to_numpy(dtype=float)

    # Add constant
    X = np.column_stack([np.ones(len(X)), X])

    try:
        # Use pseudo-inverse for numerical stability
        beta_hat = np.linalg.pinv(X.T @ X) @ X.T @ y

        dq_raw = beta_hat.T @ X.T @ X @ beta_hat
        dq_stat = float(dq_raw.item() / (float(alpha) * (1 - float(alpha))))

        dq_df = X.shape[1]
        dq_pvalue = float(1 - chi2.cdf(dq_stat, df=dq_df))

    except Exception:
        dq_stat = np.nan
        dq_df = X.shape[1]
        dq_pvalue = np.nan

    return {
        "dq_lags": lags,
        "dq_stat": dq_stat,
        "dq_df": dq_df,
        "dq_pvalue": dq_pvalue,
        "dq_pass_5pct": bool(dq_pvalue > 0.05) if pd.notna(dq_pvalue) else False
    }


def expected_shortfall_summary(actual_return, var_forecast, violations, position):
    """
    Empirical ES and tail-event severity measures.
    """
    actual = np.asarray(actual_return, dtype=float)
    var = np.asarray(var_forecast, dtype=float)
    v = np.asarray(violations).astype(bool)

    if v.sum() == 0:
        return {
            "empirical_es": np.nan,
            "avg_exceedance": np.nan,
            "max_exceedance": np.nan,
            "average_multiple_tail_event_to_var": np.nan
        }

    tail_returns = actual[v]
    tail_var = var[v]

    if position == "long":
        # VaR and returns are usually negative.
        exceedance = tail_var - tail_returns
        empirical_es = tail_returns.mean()

        # Tail loss magnitude divided by VaR loss magnitude.
        tail_loss_magnitude = np.abs(tail_returns)
        var_loss_magnitude = np.abs(tail_var)

    elif position == "short":
        # VaR and returns are usually positive.
        exceedance = tail_returns - tail_var
        empirical_es = tail_returns.mean()

        tail_loss_magnitude = np.abs(tail_returns)
        var_loss_magnitude = np.abs(tail_var)

    else:
        raise ValueError("position must be long or short")

    average_multiple = np.mean(
        tail_loss_magnitude / np.maximum(var_loss_magnitude, 1e-12)
    )

    return {
        "empirical_es": empirical_es,
        "avg_exceedance": exceedance.mean(),
        "max_exceedance": exceedance.max(),
        "average_multiple_tail_event_to_var": average_multiple
    }


def summarize_backtests(var_forecasts_df, dq_lags=5):
    """
    Summarize failure rate, Kupiec, Christoffersen, DQ and ES
    for each index-model-alpha-position combination.
    """
    rows = []

    group_cols = ["index", "model", "distribution", "var_level", "position"]

    for keys, g in var_forecasts_df.groupby(group_cols):
        index_name, model_name, dist_name, alpha, position = keys

        g = g.sort_values("date").copy()

        violations = g["violation"].astype(int).to_numpy()
        actual = g["actual_return"].to_numpy()
        varf = g["var_forecast"].to_numpy()

        kupiec = kupiec_test(violations, alpha)
        christ = christoffersen_test(violations, alpha)
        dq = dq_test(violations, varf, alpha, lags=dq_lags)
        es = expected_shortfall_summary(actual, varf, violations, position)

        row = {
            "index": index_name,
            "model": model_name,
            "distribution": dist_name,
            "var_level": alpha,
            "position": position,
        }

        row.update(kupiec)
        row.update(christ)
        row.update(dq)
        row.update(es)

        row["abs_failure_rate_error"] = abs(row["failure_rate"] - alpha)

        # Pass rule: main pass = Kupiec + Christoffersen conditional coverage
        row["main_backtest_pass"] = (
            row["kupiec_pass_5pct"]
            and row["christoffersen_cc_pass_5pct"]
        )

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# STEP 4A: IN-SAMPLE VaR BACKTESTING
# ============================================================

insample_var_rows = []
insample_fit_results = {}

for index_name in INDEX_COLUMNS:
    y = returns[index_name].dropna()

    print(f"\n==============================")
    print(f"Step 4A: In-sample VaR | {index_name}")
    print(f"==============================")

    insample_fit_results[index_name] = {}

    for model_name in MODELS_TO_RUN:
        spec = MODEL_SPECS[model_name]
        dist_name = spec["dist"]

        print(f"Fitting full-sample model: {model_name}")

        try:
            res = fit_one_model(y, spec)
            insample_fit_results[index_name][model_name] = res

            params = res.params

            # In-sample conditional mean = actual return - residual
            actual = y.loc[res.resid.index]
            resid = res.resid
            cond_mean = actual - resid
            cond_vol = res.conditional_volatility

            temp = pd.DataFrame({
                "actual_return": actual,
                "mu_forecast": cond_mean,
                "sigma_forecast": cond_vol
            }).dropna()

            for alpha in VAR_LEVELS:
                q_left = get_distribution_quantile(params, dist_name, alpha)
                q_right = get_distribution_quantile(params, dist_name, 1 - alpha)

                temp_alpha = temp.copy()

                # Long VaR: left tail
                temp_alpha["var_forecast"] = (
                    temp_alpha["mu_forecast"] + q_left * temp_alpha["sigma_forecast"]
                )
                temp_alpha["violation"] = (
                    temp_alpha["actual_return"] < temp_alpha["var_forecast"]
                ).astype(int)

                for date, row in temp_alpha.iterrows():
                    insample_var_rows.append({
                        "date": date,
                        "index": index_name,
                        "model": model_name,
                        "distribution": dist_name,
                        "var_level": alpha,
                        "position": "long",
                        "actual_return": row["actual_return"],
                        "mu_forecast": row["mu_forecast"],
                        "sigma_forecast": row["sigma_forecast"],
                        "var_forecast": row["var_forecast"],
                        "violation": row["violation"],
                        "quantile": q_left,
                        "converged": res.convergence_flag == 0
                    })

                # Short VaR: right tail
                temp_alpha = temp.copy()
                temp_alpha["var_forecast"] = (
                    temp_alpha["mu_forecast"] + q_right * temp_alpha["sigma_forecast"]
                )
                temp_alpha["violation"] = (
                    temp_alpha["actual_return"] > temp_alpha["var_forecast"]
                ).astype(int)

                for date, row in temp_alpha.iterrows():
                    insample_var_rows.append({
                        "date": date,
                        "index": index_name,
                        "model": model_name,
                        "distribution": dist_name,
                        "var_level": alpha,
                        "position": "short",
                        "actual_return": row["actual_return"],
                        "mu_forecast": row["mu_forecast"],
                        "sigma_forecast": row["sigma_forecast"],
                        "var_forecast": row["var_forecast"],
                        "violation": row["violation"],
                        "quantile": q_right,
                        "converged": res.convergence_flag == 0
                    })

            print(f"Done: {model_name}")

        except Exception as e:
            print(f"FAILED: {index_name} | {model_name} | {e}")

insample_var_forecasts = pd.DataFrame(insample_var_rows)

insample_var_forecasts.to_csv(
    VAR_DIR / "step4A_insample_var_forecasts.csv",
    index=False
)

print("In-sample VaR forecasts saved.")
print(insample_var_forecasts.shape)
display(insample_var_forecasts.head())

In [ ]:
# ============================================================
# STEP 4A: IN-SAMPLE BACKTESTING SUMMARY
# ============================================================

insample_backtest_summary = summarize_backtests(
    insample_var_forecasts,
    dq_lags=5
)

insample_backtest_summary.to_csv(
    VAR_DIR / "step4A_insample_var_backtesting_summary.csv",
    index=False
)

display(
    insample_backtest_summary[
        [
            "index", "model", "distribution", "var_level", "position",
            "n_forecasts", "n_violations", "expected_violations",
            "failure_rate", "abs_failure_rate_error",
            "kupiec_pvalue", "kupiec_pass_5pct",
            "christoffersen_cc_pvalue", "christoffersen_cc_pass_5pct",
            "dq_pvalue", "dq_pass_5pct",
            "empirical_es", "avg_exceedance",
            "main_backtest_pass", 
        ]
    ].round(6)
)

In [ ]:
# ============================================================
# STEP 4B: OUT-OF-SAMPLE ROLLING VaR FORECASTING
# ============================================================

WINDOW_SIZE = 1000

# REFIT_EVERY:
# 1  = refit model every day, slowest but strictest
# 20 = refit approximately monthly, much faster and acceptable for assignment
# 60 = faster robustness / testing mode
REFIT_EVERY = 20

# Use None for full rolling period.
# Use e.g. 250 for a quick test before full run.
MAX_FORECASTS = None

print("Rolling settings:")
print("WINDOW_SIZE:", WINDOW_SIZE)
print("REFIT_EVERY:", REFIT_EVERY)
print("MAX_FORECASTS:", MAX_FORECASTS)

In [ ]:
# ============================================================
# ROLLING FORECAST FUNCTION
# ============================================================

def rolling_var_forecast_one_series(
    y,
    index_name,
    model_name,
    spec,
    window_size=1000,
    refit_every=20,
    max_forecasts=None,
    var_levels=[0.01, 0.05]
):
    """
    Rolling one-day-ahead VaR forecast.

    Forecast y[t] using information up to y[t-1].
    Parameters are refitted every refit_every days.
    Between refits, fixed parameters are used but volatility is updated
    using the latest rolling window.
    """
    y = y.dropna().sort_index()

    forecast_positions = list(range(window_size, len(y)))

    if max_forecasts is not None:
        forecast_positions = forecast_positions[:max_forecasts]

    rows = []

    last_params = None
    last_fit_success = False
    last_convergence_flag = np.nan
    last_optimization_message = None

    dist_name = spec["dist"]

    for k, t in enumerate(forecast_positions):
        forecast_date = y.index[t]
        actual_return = y.iloc[t]

        train = y.iloc[t - window_size:t]

        need_refit = (
            last_params is None
            or k % refit_every == 0
            or not last_fit_success
        )

        try:
            am = make_arch_model(train, spec)

            if need_refit:
                res = am.fit(
                    update_freq=0,
                    disp="off",
                    show_warning=False,
                    options={"maxiter": 2000}
                )

                last_params = res.params
                last_fit_success = (res.convergence_flag == 0)
                last_convergence_flag = res.convergence_flag
                last_optimization_message = str(res.optimization_result.message)

            else:
                # Use fixed last parameters but update state using latest data window
                res = am.fix(last_params)

            forecast = res.forecast(horizon=1, reindex=False)

            mu_forecast = float(forecast.mean.iloc[-1, 0])
            variance_forecast = float(forecast.variance.iloc[-1, 0])
            sigma_forecast = np.sqrt(max(variance_forecast, 0))

            for alpha in var_levels:
                q_left = get_distribution_quantile(last_params, dist_name, alpha)
                q_right = get_distribution_quantile(last_params, dist_name, 1 - alpha)

                var_long = mu_forecast + q_left * sigma_forecast
                var_short = mu_forecast + q_right * sigma_forecast

                rows.append({
                    "date": forecast_date,
                    "index": index_name,
                    "model": model_name,
                    "distribution": dist_name,
                    "var_level": alpha,
                    "position": "long",
                    "actual_return": actual_return,
                    "mu_forecast": mu_forecast,
                    "sigma_forecast": sigma_forecast,
                    "var_forecast": var_long,
                    "violation": int(actual_return < var_long),
                    "quantile": q_left,
                    "window_size": window_size,
                    "refit_every": refit_every,
                    "forecast_number": k + 1,
                    "refit_on_this_date": need_refit,
                    "convergence_flag": last_convergence_flag,
                    "fit_success": last_fit_success,
                    "optimization_message": last_optimization_message
                })

                rows.append({
                    "date": forecast_date,
                    "index": index_name,
                    "model": model_name,
                    "distribution": dist_name,
                    "var_level": alpha,
                    "position": "short",
                    "actual_return": actual_return,
                    "mu_forecast": mu_forecast,
                    "sigma_forecast": sigma_forecast,
                    "var_forecast": var_short,
                    "violation": int(actual_return > var_short),
                    "quantile": q_right,
                    "window_size": window_size,
                    "refit_every": refit_every,
                    "forecast_number": k + 1,
                    "refit_on_this_date": need_refit,
                    "convergence_flag": last_convergence_flag,
                    "fit_success": last_fit_success,
                    "optimization_message": last_optimization_message
                })

        except Exception as e:
            rows.append({
                "date": forecast_date,
                "index": index_name,
                "model": model_name,
                "distribution": dist_name,
                "var_level": np.nan,
                "position": "failed",
                "actual_return": actual_return,
                "mu_forecast": np.nan,
                "sigma_forecast": np.nan,
                "var_forecast": np.nan,
                "violation": np.nan,
                "quantile": np.nan,
                "window_size": window_size,
                "refit_every": refit_every,
                "forecast_number": k + 1,
                "refit_on_this_date": need_refit,
                "convergence_flag": np.nan,
                "fit_success": False,
                "optimization_message": str(e)
            })

            last_fit_success = False
            last_params = None

    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# RUN STEP 4B ROLLING FORECASTS
# ============================================================

rolling_var_results = []

for index_name in INDEX_COLUMNS:
    y = returns[index_name].dropna()

    print(f"\n==============================")
    print(f"Step 4B rolling VaR | {index_name}")
    print(f"==============================")

    for model_name in MODELS_TO_RUN:
        spec = MODEL_SPECS[model_name]

        print(f"Running rolling VaR: {index_name} | {model_name}")

        df_model = rolling_var_forecast_one_series(
            y=y,
            index_name=index_name,
            model_name=model_name,
            spec=spec,
            window_size=WINDOW_SIZE,
            refit_every=REFIT_EVERY,
            max_forecasts=MAX_FORECASTS,
            var_levels=VAR_LEVELS
        )

        rolling_var_results.append(df_model)

        # Save each model result immediately to avoid losing progress
        safe_name = f"step4B_rolling_var_{index_name}_{model_name}.csv"
        df_model.to_csv(VAR_DIR / safe_name, index=False)

        n_failed = (df_model["position"] == "failed").sum()
        print(f"Done | rows={len(df_model)} | failed rows={n_failed}")

rolling_var_forecasts = pd.concat(rolling_var_results, ignore_index=True)

# Remove failed rows from main backtest dataset
rolling_var_forecasts_clean = rolling_var_forecasts[
    rolling_var_forecasts["position"].isin(["long", "short"])
].copy()

rolling_var_forecasts.to_csv(
    VAR_DIR / "step4B_rolling_var_forecasts_all_rows.csv",
    index=False
)

rolling_var_forecasts_clean.to_csv(
    VAR_DIR / "step4B_rolling_var_forecasts_clean.csv",
    index=False
)

print("Rolling VaR forecast complete.")
print("All rows:", rolling_var_forecasts.shape)
print("Clean rows:", rolling_var_forecasts_clean.shape)

display(rolling_var_forecasts_clean.head())

In [ ]:
# ============================================================
# STEP 4B: OUT-OF-SAMPLE BACKTESTING SUMMARY
# ============================================================

rolling_backtest_summary = summarize_backtests(
    rolling_var_forecasts_clean,
    dq_lags=5
)

rolling_backtest_summary.to_csv(
    VAR_DIR / "step4B_rolling_var_backtesting_summary.csv",
    index=False
)

display(
    rolling_backtest_summary[
        [
            "index", "model", "distribution", "var_level", "position",
            "n_forecasts", "n_violations", "expected_violations",
            "failure_rate", "abs_failure_rate_error",
            "kupiec_pvalue", "kupiec_pass_5pct",
            "christoffersen_cc_pvalue", "christoffersen_cc_pass_5pct",
            "dq_pvalue", "dq_pass_5pct",
            "empirical_es", "avg_exceedance",
            "main_backtest_pass", "average_multiple_tail_event_to_var"
        ]
    ].round(6)
)

In [ ]:
# ============================================================
# COMPARE IN-SAMPLE VS OUT-OF-SAMPLE BACKTESTING
# ============================================================

insample_compare = insample_backtest_summary.copy()
insample_compare["sample_type"] = "in_sample"

rolling_compare = rolling_backtest_summary.copy()
rolling_compare["sample_type"] = "out_of_sample_rolling"

combined_backtest_summary = pd.concat(
    [insample_compare, rolling_compare],
    ignore_index=True
)

combined_backtest_summary.to_csv(
    VAR_DIR / "step4_combined_insample_outsample_backtesting_summary.csv",
    index=False
)

comparison_cols = [
    "sample_type", "index", "model", "distribution", "var_level", "position",
    "n_forecasts", "n_violations", "expected_violations",
    "failure_rate", "abs_failure_rate_error",
    "kupiec_pvalue", "kupiec_pass_5pct",
    "christoffersen_cc_pvalue", "christoffersen_cc_pass_5pct",
    "dq_pvalue", "dq_pass_5pct",
    "empirical_es", "avg_exceedance",
    "main_backtest_pass"
]

display(combined_backtest_summary[comparison_cols].round(6))

In [ ]:
# ============================================================
# MODEL RANKING BASED ON OUT-OF-SAMPLE VaR BACKTESTING
# ============================================================

ranking = rolling_backtest_summary.copy()

# Convert booleans to integers
ranking["kupiec_pass_int"] = ranking["kupiec_pass_5pct"].astype(int)
ranking["christoffersen_cc_pass_int"] = ranking["christoffersen_cc_pass_5pct"].astype(int)
ranking["dq_pass_int"] = ranking["dq_pass_5pct"].astype(int)
ranking["main_pass_int"] = ranking["main_backtest_pass"].astype(int)

# Rank within each index-position-var_level
ranking = ranking.sort_values(
    by=[
        "index",
        "position",
        "var_level",
        "main_pass_int",
        "kupiec_pass_int",
        "christoffersen_cc_pass_int",
        "dq_pass_int",
        "abs_failure_rate_error",
        "avg_exceedance"
    ],
    ascending=[
        True,
        True,
        True,
        False,
        False,
        False,
        False,
        True,
        True
    ]
)

ranking["rank_within_index_position_level"] = (
    ranking
    .groupby(["index", "position", "var_level"])
    .cumcount() + 1
)

best_models_by_index_position_level = ranking[
    ranking["rank_within_index_position_level"] == 1
].copy()

ranking.to_csv(
    VAR_DIR / "step4B_model_ranking_outsample_var.csv",
    index=False
)

best_models_by_index_position_level.to_csv(
    VAR_DIR / "step4B_best_models_by_index_position_level.csv",
    index=False
)

display(
    best_models_by_index_position_level[
        [
            "index", "position", "var_level", "model", "distribution",
            "failure_rate", "abs_failure_rate_error",
            "kupiec_pvalue", "christoffersen_cc_pvalue", "dq_pvalue",
            "main_backtest_pass",
            "rank_within_index_position_level"
        ]
    ].round(6)
)

In [ ]:
# ============================================================
# AGGREGATE MODEL PERFORMANCE ACROSS INDICES / POSITIONS / LEVELS
# ============================================================

model_overall_score = (
    rolling_backtest_summary
    .groupby(["model", "distribution"])
    .agg(
        n_tests=("main_backtest_pass", "count"),
        main_pass_count=("main_backtest_pass", "sum"),
        kupiec_pass_count=("kupiec_pass_5pct", "sum"),
        christoffersen_pass_count=("christoffersen_cc_pass_5pct", "sum"),
        dq_pass_count=("dq_pass_5pct", "sum"),
        avg_abs_failure_rate_error=("abs_failure_rate_error", "mean"),
        avg_failure_rate=("failure_rate", "mean"),
        avg_exceedance=("avg_exceedance", "mean")
    )
    .reset_index()
)

model_overall_score["main_pass_rate"] = (
    model_overall_score["main_pass_count"] / model_overall_score["n_tests"]
)

model_overall_score["kupiec_pass_rate"] = (
    model_overall_score["kupiec_pass_count"] / model_overall_score["n_tests"]
)

model_overall_score["christoffersen_pass_rate"] = (
    model_overall_score["christoffersen_pass_count"] / model_overall_score["n_tests"]
)

model_overall_score["dq_pass_rate"] = (
    model_overall_score["dq_pass_count"] / model_overall_score["n_tests"]
)

model_overall_score = model_overall_score.sort_values(
    by=[
        "main_pass_rate",
        "kupiec_pass_rate",
        "christoffersen_pass_rate",
        "dq_pass_rate",
        "avg_abs_failure_rate_error"
    ],
    ascending=[
        False,
        False,
        False,
        False,
        True
    ]
)

model_overall_score.to_csv(
    VAR_DIR / "step4B_overall_model_performance_score.csv",
    index=False
)

display(model_overall_score.round(6))

In [ ]:
# ============================================================
# SIZE-SEGMENT RISK COMPARISON
# Use average absolute VaR magnitude and failure patterns
# ============================================================

risk_rows = []

for (index_name, model_name, alpha, position), g in rolling_var_forecasts_clean.groupby(
    ["index", "model", "var_level", "position"]
):
    g = g.copy()

    if position == "long":
        avg_var_magnitude = abs(g["var_forecast"].mean())
        avg_tail_var = abs(g["var_forecast"].quantile(0.05))
    else:
        avg_var_magnitude = abs(g["var_forecast"].mean())
        avg_tail_var = abs(g["var_forecast"].quantile(0.95))

    risk_rows.append({
        "index": index_name,
        "model": model_name,
        "var_level": alpha,
        "position": position,
        "avg_var_forecast": g["var_forecast"].mean(),
        "avg_abs_var_forecast": avg_var_magnitude,
        "avg_tail_var_reference": avg_tail_var,
        "failure_rate": g["violation"].mean()
    })

size_risk_table = pd.DataFrame(risk_rows)

size_risk_summary = (
    size_risk_table
    .groupby(["index", "var_level", "position"])
    .agg(
        avg_abs_var_across_models=("avg_abs_var_forecast", "mean"),
        avg_failure_rate_across_models=("failure_rate", "mean")
    )
    .reset_index()
)

size_risk_summary["risk_rank_by_avg_abs_var"] = (
    size_risk_summary
    .groupby(["var_level", "position"])["avg_abs_var_across_models"]
    .rank(ascending=False, method="dense")
)

size_risk_table.to_csv(
    VAR_DIR / "step4B_size_segment_risk_by_model.csv",
    index=False
)

size_risk_summary.to_csv(
    VAR_DIR / "step4B_size_segment_risk_summary.csv",
    index=False
)

display(size_risk_summary.round(6))

In [ ]:
# ============================================================
# PLOT ACTUAL RETURNS VS VaR BANDS
# ============================================================

plot_level = 0.01

for index_name in INDEX_COLUMNS:
    best_for_index = best_models_by_index_position_level[
        (best_models_by_index_position_level["index"] == index_name)
        & (best_models_by_index_position_level["var_level"] == plot_level)
    ]

    # Prefer long best model for the plot
    if len(best_for_index[best_for_index["position"] == "long"]) > 0:
        model_name = best_for_index[best_for_index["position"] == "long"]["model"].iloc[0]
    else:
        model_name = "AR1_APARCH11_SkewT"

    g_long = rolling_var_forecasts_clean[
        (rolling_var_forecasts_clean["index"] == index_name)
        & (rolling_var_forecasts_clean["model"] == model_name)
        & (rolling_var_forecasts_clean["var_level"] == plot_level)
        & (rolling_var_forecasts_clean["position"] == "long")
    ].copy()

    g_short = rolling_var_forecasts_clean[
        (rolling_var_forecasts_clean["index"] == index_name)
        & (rolling_var_forecasts_clean["model"] == model_name)
        & (rolling_var_forecasts_clean["var_level"] == plot_level)
        & (rolling_var_forecasts_clean["position"] == "short")
    ].copy()

    if g_long.empty or g_short.empty:
        continue

    g = g_long[["date", "actual_return", "var_forecast", "violation"]].rename(
        columns={
            "var_forecast": "var_long",
            "violation": "long_violation"
        }
    )

    g = g.merge(
        g_short[["date", "var_forecast", "violation"]].rename(
            columns={
                "var_forecast": "var_short",
                "violation": "short_violation"
            }
        ),
        on="date",
        how="inner"
    )

    plt.figure(figsize=(14, 5))
    plt.plot(g["date"], g["actual_return"], linewidth=0.8, label="Actual return")
    plt.plot(g["date"], g["var_long"], linewidth=1.0, label=f"Long VaR {int(plot_level*100)}%")
    plt.plot(g["date"], g["var_short"], linewidth=1.0, label=f"Short VaR {int(plot_level*100)}%")

    long_breach = g[g["long_violation"] == 1]
    short_breach = g[g["short_violation"] == 1]

    plt.scatter(
        long_breach["date"],
        long_breach["actual_return"],
        s=12,
        label="Long VaR violation"
    )

    plt.scatter(
        short_breach["date"],
        short_breach["actual_return"],
        s=12,
        label="Short VaR violation"
    )

    plt.title(f"{index_name}: actual returns vs VaR bands | {model_name}")
    plt.xlabel("Date")
    plt.ylabel("Return / VaR (%)")
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        FIGURE_DIR / f"step4B_{index_name}_{model_name}_actual_vs_var_{int(plot_level*100)}pct.png",
        dpi=300
    )

    plt.show()

In [ ]:
# ============================================================
# EXPORT STEP 4 RESULTS TO EXCEL
# ============================================================

excel_path = VAR_DIR / "step4_var_backtesting_results.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    insample_backtest_summary.to_excel(writer, sheet_name="4A_InSample_Backtest", index=False)
    rolling_backtest_summary.to_excel(writer, sheet_name="4B_Rolling_Backtest", index=False)
    combined_backtest_summary.to_excel(writer, sheet_name="Combined_Backtest", index=False)
    ranking.to_excel(writer, sheet_name="4B_Model_Ranking", index=False)
    best_models_by_index_position_level.to_excel(writer, sheet_name="4B_Best_Models", index=False)
    model_overall_score.to_excel(writer, sheet_name="4B_Overall_Model_Score", index=False)
    size_risk_summary.to_excel(writer, sheet_name="Size_Risk_Summary", index=False)
    size_risk_table.to_excel(writer, sheet_name="Size_Risk_By_Model", index=False)

print("Saved Step 4 Excel report:")
print(excel_path)

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

FIG_DIR = Path("figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Optional: make plots publication-friendly
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.grid": True,
    "grid.alpha": 0.3,
})


INDEX_ORDER = ["VN30", "VNAllshare", "VNMidcap", "VNSmallcap"]


def find_date_column(df):
    candidates = ["Date", "date", "trading_date", "TradingDate", "time", "Time"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"No date column found. Available columns: {df.columns.tolist()}")


def standardize_date_index(df):
    df = df.copy()
    date_col = find_date_column(df)
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).set_index(date_col)
    return df


def pick_index_columns(df):
    cols = []
    for idx in INDEX_ORDER:
        exact = [c for c in df.columns if c == idx]
        contains = [c for c in df.columns if idx.lower() in c.lower()]
        if exact:
            cols.append(exact[0])
        elif contains:
            cols.append(contains[0])
        else:
            raise ValueError(f"Cannot find column for {idx}. Available columns: {df.columns.tolist()}")
    return cols


def savefig(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"Saved: {path}")

In [ ]:
# ============================================================
# Figure 4: Overall out-of-sample main pass rates
# Input: step4B_overall_model_performance_score.csv
# Output: figures/fig_04_oos_main_pass_rate.pdf
# ============================================================

score_path = Path("outputs/var_backtesting/step4B_overall_model_performance_score.csv")
score = pd.read_csv(score_path)

# Flexible column handling
model_col = "model" if "model" in score.columns else "Model"
dist_col = "distribution" if "distribution" in score.columns else (
    "dist" if "dist" in score.columns else "Dist."
)

main_col_candidates = [
    "main_pass_rate", "Main pass", "main_pass", "main_pass_pct", "main_pass_rate_pct"
]
main_col = next((c for c in main_col_candidates if c in score.columns), None)
if main_col is None:
    raise ValueError(f"Cannot find main pass rate column. Available columns: {score.columns.tolist()}")

plot_df = score.copy()

# Build model label
plot_df["label"] = plot_df[model_col].astype(str)
if dist_col in plot_df.columns:
    plot_df["label"] = plot_df["label"] + " - " + plot_df[dist_col].astype(str)

# Convert pass rate to percentage if stored as fraction
plot_df["main_pass_rate_plot"] = pd.to_numeric(plot_df[main_col], errors="coerce")
if plot_df["main_pass_rate_plot"].max() <= 1.0:
    plot_df["main_pass_rate_plot"] *= 100

plot_df = plot_df.sort_values("main_pass_rate_plot", ascending=True)

fig, ax = plt.subplots(figsize=(7.2, 4.8))

ax.barh(
    plot_df["label"],
    plot_df["main_pass_rate_plot"],
    height=0.65,
    zorder=2
)

ax.set_xlabel("Main pass rate (%)")
ax.set_ylabel("")

# No title inside the figure; use LaTeX caption instead
ax.grid(False)


# Set x-axis limit with enough room for value labels
x_max = max(100, plot_df["main_pass_rate_plot"].max() * 1.15)
ax.set_xlim(0, x_max)

# Percentage labels
for y, v in enumerate(plot_df["main_pass_rate_plot"]):
    ax.text(
        v + x_max * 0.01,
        y,
        f"{v:.1f}%",
        va="center",
        fontsize=8
    )

ax.tick_params(axis="both", labelsize=9)

fig.tight_layout()

savefig(FIG_DIR / "fig_04_oos_main_pass_rate.pdf")

In [ ]:
# ============================================================
# FIGURE 5 FOR REPORT:
# Actual returns with 1% long VaR bands and violations
# VN30 and VNAllshare, EGARCH-SkewT
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use the same output folder as the rest of Step 4
FIGURE_DIR = Path("outputs/figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

VAR_DIR = Path("outputs/var_backtesting")

# ------------------------------------------------------------
# Load rolling forecast data if not already available in memory
# ------------------------------------------------------------

try:
    df_plot_source = rolling_var_forecasts_clean.copy()
    print("Using rolling_var_forecasts_clean from notebook memory.")
except NameError:
    forecast_path = VAR_DIR / "step4B_rolling_var_forecasts_clean.csv"
    df_plot_source = pd.read_csv(forecast_path)
    print(f"Loaded rolling forecasts from: {forecast_path}")

df_plot_source["date"] = pd.to_datetime(df_plot_source["date"])

# ------------------------------------------------------------
# Select plot configuration
# ------------------------------------------------------------

plot_indices = ["VN30", "VNAllshare"]
plot_model = "AR1_EGARCH11_SkewT"
plot_level = 0.01
plot_position = "long"

plot_df = df_plot_source[
    (df_plot_source["index"].isin(plot_indices))
    & (df_plot_source["model"] == plot_model)
    & (df_plot_source["var_level"] == plot_level)
    & (df_plot_source["position"] == plot_position)
].copy()

if plot_df.empty:
    raise ValueError(
        "No data found for VN30/VNAllshare, AR1_EGARCH11_SkewT, "
        "1% long VaR. Check model names in rolling_var_forecasts_clean."
    )

# Make sure violation is integer
plot_df["violation"] = plot_df["violation"].astype(int)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(10, 5.8),
    sharex=True
)

for ax, index_name in zip(axes, plot_indices):
    g = plot_df[plot_df["index"] == index_name].sort_values("date").copy()

    ax.plot(
        g["date"],
        g["actual_return"],
        linewidth=0.65,
        label="Actual return",
        zorder=2
    )

    ax.plot(
        g["date"],
        g["var_forecast"],
        linewidth=1.1,
        linestyle="--",
        label="1% long VaR",
        zorder=3
    )

    breaches = g[g["violation"] == 1]

    ax.scatter(
        breaches["date"],
        breaches["actual_return"],
        s=18,
        marker="o",
        label="VaR violation",
        zorder=5
    )

    ax.axhline(
        0,
        linewidth=0.8,
        color="black",
        alpha=0.75,
        zorder=1
    )

    # Keep subplot title, remove overall title
    ax.set_title(index_name, loc="left", fontsize=10, fontweight="normal")
    ax.set_ylabel("Return / VaR (%)")

    # Remove grid for report-style figure
    ax.grid(False)


    # Clean legend
    ax.legend(loc="lower left", ncol=3, frameon=False, fontsize=8)

axes[-1].set_xlabel("Date")

# No suptitle; use LaTeX caption instead
fig.tight_layout()

# Save as PDF for LaTeX and PNG for quick checking
pdf_path = FIGURE_DIR / "fig_05_var_bands_vn30_vnallshare.pdf"
png_path = FIGURE_DIR / "fig_05_var_bands_vn30_vnallshare.png"

plt.savefig(pdf_path, bbox_inches="tight")
plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved PDF: {pdf_path}")
print(f"Saved PNG: {png_path}")

# ------------------------------------------------------------
# Optional: print violation counts for text interpretation
# ------------------------------------------------------------

summary = (
    plot_df
    .groupby("index")
    .agg(
        n_forecasts=("violation", "count"),
        n_violations=("violation", "sum"),
        failure_rate=("violation", "mean")
    )
    .reset_index()
)

summary["failure_rate_pct"] = summary["failure_rate"] * 100

display(summary)